# ASR - LLM pipeline on Earnings21

1. Run Parakeet ASR on short audio chunks
2. Ask LLM to select suspicious sentences
3. Correct the selected chunks manually
4. Use the reviewed chunks as examples for LLM correction
5. Compare ASR and corrected text with the reference transcript (GT)

In [ ]:
!pip -q install 'protobuf>=5.26.1,<6.0.0' nemo_toolkit[asr] transformers accelerate jiwer huggingface_hub

import json
import re
import tempfile
import wave
from pathlib import Path

import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Torch: 2.11.0+cu128
CUDA available: True


Download earnings21 into your Google Drive from https://huggingface.co/datasets/Revai/earnings21/tree/main

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = Path('/content/drive/MyDrive/earnings21')
OUTPUT_FILE = Path('/content/drive/MyDrive/earnings21_records.jsonl')

print('WAV files count:', len(list((DATASET_DIR / 'wav').glob('*.wav'))))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WAV files count: 44


In [ ]:
# increase to process more recordings at once
CHUNK_SECONDS = 30
BATCH_SIZE = 1

# take the second wav file
audio_files = sorted((DATASET_DIR / 'wav').glob('*.wav'))[1:2]

if not audio_files:
  raise FileNotFoundError()

print('Selected recordings:', [p.name for p in audio_files])

Selected recordings: ['4330115.wav']


In [ ]:
def split_wav(audio_file, max_seconds, output_dir, max_chunks=None):
    chunks = []
    with wave.open(str(audio_file), 'rb') as source:
        frame_rate = source.getframerate()
        frames_per_chunk = -1 if max_seconds is None else frame_rate * max_seconds
        index = 0
        while max_chunks is None or index < max_chunks:
            frames = source.readframes(frames_per_chunk)
            if not frames:
                break
            chunk_file = output_dir / f'{audio_file.stem}_{index:04d}.wav'
            with wave.open(str(chunk_file), 'wb') as target:
                target.setnchannels(source.getnchannels())
                target.setsampwidth(source.getsampwidth())
                target.setframerate(frame_rate)
                target.writeframes(frames)
            chunks.append(chunk_file)
            index += 1
    return chunks

def load_reference(audio_file):
    reference_file = DATASET_DIR / 'nlp_references' / f'{audio_file.stem}.nlp'
    words = []
    with reference_file.open(encoding='utf-8') as file:
        for line in file:
            fields = line.rstrip('\n').split('|')
            if fields and fields[0] and fields[0] != 'token':
                punctuation = fields[4] if len(fields) > 4 else ''
                words.append(fields[0] + punctuation)
    return ' '.join(words)

In [ ]:
import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.ASRModel.from_pretrained(
    model_name='nvidia/parakeet-tdt-0.6b-v3'
)

asr_records = []

with tempfile.TemporaryDirectory(prefix='earnings21_chunks-') as directory:
    chunk_dir = Path(directory)

    chunk_files = [
        (audio_file, chunk)
        for audio_file in audio_files
        for chunk in split_wav(
            audio_file,
            CHUNK_SECONDS,
            chunk_dir
        )
    ]

    for start in range(0, len(chunk_files), BATCH_SIZE):
        batch = chunk_files[start:start + BATCH_SIZE]
        batch_paths = [chunk_path for _, chunk_path in batch]

        outputs = asr_model.transcribe(
            [str(path) for path in batch_paths],
            batch_size=BATCH_SIZE,
            num_workers=0
        )

        for (source_file, chunk_file), output in zip(batch, outputs):
            asr_records.append({
                'id': chunk_file.stem,
                'source_file': source_file.name,
                'chunk_file': chunk_file.name,
                'asr_text': output.text.strip(),
                'selected': False,
                'human_text': None,
                'corrected_text': None,
            })

        print(
            f"Transcribed "
            f"{min(start + BATCH_SIZE, len(chunk_files))}/"
            f"{len(chunk_files)} chunks"
        )

reference_text = load_reference(audio_files[0])

print(f"ASR chunks: {len(asr_records)}")
print(f"Reference words: {len(reference_text.split())}")

del asr_model
torch.cuda.empty_cache()

[NeMo I 2026-09-30 23:22:44 mixins:194] Tokenizer SentencePieceTokenizer initialized with 8192 tokens


[NeMo W 2026-09-30 23:22:48 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 10.0
    min_duration: 1.0
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-09-30 23:22:48 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation 

[NeMo I 2026-09-30 23:22:56 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-09-30 23:22:56 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-09-30 23:22:56 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-09-30 23:23:06 save_restore_connector:287] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v3/snapshots/541d1f99c6b0c3cd0b11a95167540bb8edefd82b/parakeet-tdt-0.6b-v3.nemo.


[NeMo W 2026-09-30 23:23:09 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-30 23:23:09 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:01,  1.99s/it]
[NeMo W 2026-09-30 23:23:11 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-30 23:23:11 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tok

Transcribed 1/82 chunks


...

Transcribed 80/82 chunks


Transcribing: 1it [00:00,  3.98it/s]
[NeMo W 2026-09-30 23:23:42 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-09-30 23:23:42 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


Transcribed 81/82 chunks


Transcribing: 1it [00:00,  3.85it/s]


Transcribed 82/82 chunks
ASR chunks: 82
Reference words: 6602


## Load Gemma on the Colab GPU

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError()

LLM_NAME = 'google/gemma-4-E2B-it'
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

print('Loaded:', LLM_NAME)
print('Model device:', next(llm.parameters()).device)

def generate(instruction, max_new_tokens=128):
    messages = [
        {'role': 'system', 'content': 'Follow the task exactly. Return only the requested answer.'},
        {'role': 'user', 'content': instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# from transformers import AutoModelForCausalLM, AutoTokenizer
# from google.colab import userdata

# if not torch.cuda.is_available():
#     raise RuntimeError()

# LLM_NAME = 'meta-llama/Llama-3.2-3B-Instruct'
# HF_TOKEN = userdata.get('HF_TOKEN')

# tokenizer = AutoTokenizer.from_pretrained(LLM_NAME, token=HF_TOKEN)
# llm = AutoModelForCausalLM.from_pretrained(
#     LLM_NAME,
#     token=HF_TOKEN,
#     device_map='auto',
#     dtype=torch.float16,
# )

# print('Loaded:', LLM_NAME)
# print('Model device:', next(llm.parameters()).device)

# def generate(instruction, max_new_tokens=512):
#     messages = [
#         {'role': 'system', 'content': 'Follow the task exactly. Return only the requested answer.'},
#         {'role': 'user', 'content': instruction},
#     ]
#     inputs = tokenizer.apply_chat_template(
#         messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
#     ).to(llm.device)
#     with torch.no_grad():
#         output = llm.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     generated = output[0][inputs['input_ids'].shape[1]:]
#     return tokenizer.decode(generated, skip_special_tokens=True).strip()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Loaded: google/gemma-4-E2B-it
Model device: cuda:0


In [ ]:
# The paper used 10 manually annotated examples,
# we drop it to 5 so collab does not die

candidate_records = []

for record in asr_records:
    sentences = re.split(
        r'(?<=[.!?])\s+',
        record['asr_text'].strip()
    )

    for sentence_index, sentence in enumerate(sentences):
        sentence = sentence.strip()

        if sentence:
            candidate_records.append({
                'id': f"{record['id']}_sentence_{sentence_index:04d}",
                'source_file': record['source_file'],
                'asr_text': sentence,
                'selected': False,
                'human_text': None,
                'corrected_text': None,
            })

MAX_CANDIDATES = 40
candidate_records = candidate_records[:MAX_CANDIDATES]

transcript = "\n".join(
    f"{index}: {record['asr_text']}"
    for index, record in enumerate(candidate_records)
)

selection_prompt = (
    "List exactly 5 different sentences from the following call transcript that contain "
    "the highest number of names, numbers written in words, companies, websites."
    "Do not take them from the same part of the transcript, try to analyze the entire transcript first to find sentences that would fit the assignment.\n\n"
    "Return only the integer line indices separated by commas. "
    "Do not provide explanations or additional text.\n\n"
    + transcript
)

response = generate(selection_prompt, max_new_tokens=100)

selected = {
    int(value)
    for value in re.findall(r"\b\d+\b", response)
    if int(value) < len(candidate_records)
}

selected = set(sorted(selected)[:5])

if not selected:
    raise ValueError(f"No valid indices returned: {response!r}")

for index, record in enumerate(candidate_records):
    record['selected'] = index in selected

print("Model response:", response)
print("Selected:", sorted(selected))
print("Candidate segments:", len(candidate_records))

Model response: 17,18,20
Selected: [17, 18, 20]
Candidate segments: 40


## Human correction

In [ ]:
for record in candidate_records:
    if not record['selected']:
        continue

    print(f"\n[{record['id']}]")
    print(f"ASR: {record['asr_text']}")

    text = input(
        "Correct transcription (Enter keeps ASR): "
    ).strip()

    record['human_text'] = text or record['asr_text']

examples = [
    (
        f"ASR: {record['asr_text']}\n"
        f"Human: {record['human_text']}"
    )
    for record in candidate_records
    if record['selected'] and record['human_text']
]

if not examples:
    raise ValueError("No reviewed examples available.")



[4330115_0003_sentence_0000]
ASR: 8K filed yesterday and posted on the company's website at Culp.com.
Correct transcription (Enter keeps ASR): 8-K filed yesterday and posted on the company's website at culp.com.

[4330115_0003_sentence_0001]
ASR: A slide presentation with supporting summary financial information is also available on the company's website as part of the webcast today.
Correct transcription (Enter keeps ASR): A slide presentation with supporting summary financial information is also available on the company's website as part of the webcast today.

[4330115_0004_sentence_0000]
ASR: which is available in the AK filed yesterday and posted on the company's website.
Correct transcription (Enter keeps ASR): which is available in the 8-K filed yesterday and posted on the company's website.


In [ ]:
from jiwer import wer

# we do not accept the correction if it it's wer is too big
WER_THRESHOLD = 0.25
examples_text = "\n\n".join(examples)

for index, record in enumerate(asr_records, start=1):
    correction_prompt = (
            "Correct only clear ASR recognition errors in the sentence below. "
            "Preserve every word, repetition, contraction, number, punctuation "
            "choice, and capitalization unless it is clearly an ASR error. "
            "Do not improve grammar or style. Do not add hyphens. "
            "Do not remove repeated words unless the repetition is clearly "
            "a recognition error. Return only the corrected sentence.\n\n"
            f"Reviewed examples:\n{examples_text}\n\n"
            f"ASR transcript:\n{record['asr_text']}"
    )

    proposed_text = generate(
        correction_prompt,
        max_new_tokens=256
    ).strip()

    if not proposed_text:
        proposed_text = record['asr_text']

    chunk_wer = wer(
        record['asr_text'].lower(),
        proposed_text.lower()
    )

    accepted = chunk_wer < WER_THRESHOLD

    record['proposed_text'] = proposed_text
    record['correction_wer'] = chunk_wer
    record['correction_accepted'] = accepted
    record['corrected_text'] = (
        proposed_text if accepted else record['asr_text']
    )

    print(
        f"[{index}/{len(asr_records)}] "
        f"{record['id']} | "
        f"change WER={chunk_wer:.2%} | "
        f"accepted={accepted}"
    )

    if proposed_text != record['asr_text']:
        print("  ASR:      ", record['asr_text'])
        print("  Proposed: ", proposed_text)


[1/82] 4330115_0000 | change WER=2.53% | accepted=True
  ASR:       Good day and welcome to Culp's Third Quarter 2020 Earnings Conference Call. Today's call is being recorded. At this time for opening remarks and introductions, I'd like to turn the call over to Ms. Drew Anderson. Please go ahead. Thank you. Good morning, and welcome to the Culp Conference Call to review the company's results for the third quarter of fiscal 2020. As we start, let me state that this morning's call will contain forward-looking statements about the business.
  Proposed:  Good day and welcome to Culp's Third Quarter 2020 Earnings Conference Call. Today's call is being recorded. At this time for opening remarks and introductions, I'd like to turn the call over to Ms. Drew Anderson. Please go ahead. Thank you. Good morning, and welcome to the Culp Conference Call to review the company's results for the third quarter of fiscal 2020. As we start, let me state that this morning's call will contain forward lookin

In [ ]:
from jiwer import wer
import re

def normalize_for_wer(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

asr_full_text = " ".join(
    record["asr_text"].strip()
    for record in asr_records
)

corrected_full_text = " ".join(
    (record.get("corrected_text") or record["asr_text"]).strip()
    for record in asr_records
)

normalized_reference = normalize_for_wer(reference_text)
normalized_asr = normalize_for_wer(asr_full_text)
normalized_corrected = normalize_for_wer(corrected_full_text)

asr_wer = wer(normalized_reference, normalized_asr)
corrected_wer = wer(normalized_reference, normalized_corrected)

print("\n=== WER RESULTS ===")
print(f"Reference words: {len(normalized_reference.split())}")
print(f"ASR WER:        {asr_wer:.2%}")
print(f"Corrected WER:  {corrected_wer:.2%}")
print(f"WER improvement:{asr_wer - corrected_wer:+.2%}")

print("\n=== CHUNK DIFFERENCES ===")

different_chunks = 0

for record in asr_records:
    asr_text = record["asr_text"]
    corrected_text = record.get("corrected_text") or asr_text

    if asr_text == corrected_text:
        continue

    different_chunks += 1

    print(f"\n[{record['id']}]")
    print("ASR:       ", asr_text)
    print("Corrected: ", corrected_text)

print(f"\nChanged chunks: {different_chunks}/{len(asr_records)}")


=== WER RESULTS ===
Reference words: 6833
ASR WER:        10.10%
Corrected WER:  10.46%
WER change:     +0.37%
WER improvement:-0.37%

=== CHUNK DIFFERENCES ===

[4330115_0000]
ASR:        Good day and welcome to Culp's Third Quarter 2020 Earnings Conference Call. Today's call is being recorded. At this time for opening remarks and introductions, I'd like to turn the call over to Ms. Drew Anderson. Please go ahead. Thank you. Good morning, and welcome to the Culp Conference Call to review the company's results for the third quarter of fiscal 2020. As we start, let me state that this morning's call will contain forward-looking statements about the business.
Corrected:  Good day and welcome to Culp's Third Quarter 2020 Earnings Conference Call. Today's call is being recorded. At this time for opening remarks and introductions, I'd like to turn the call over to Ms. Drew Anderson. Please go ahead. Thank you. Good morning, and welcome to the Culp Conference Call to review the company's res